# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MihirJayswal812007/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. Unit of Analysis: One row represents a single anonymized webpage's performance over a specific month.
2. Time Window: I am using a mid-panel month, specifically month = '2026-03', to ensure I treat the final month of the dataset as a sealed test set and avoid outcome leakage.
3. Table Used: The main internship-warehouse dataset from Hugging Face.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Load HF Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect to DuckDB & pass the token directly to it securely
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. CORRECTED Path to the warehouse
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Query 1: Verify the grain and row count
query_grain = f"SELECT count(*) as total_rows FROM '{hf_path}'"
display(con.execute(query_grain).df())

,total_rows
0,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label (Proxy): needs_redesign. This is a binary label (1 if the page ranks on Page 1 but has a CTR < 0.05, else 0).
Excluded: I am excluding pages with 0 impressions. If a page is entirely invisible in search, it is an SEO issue, not a UI/UX layout issue.

My 5 Features:

1.content_type: Knowable at the decision moment because the format is static in the CMS.

2.word_count: Knowable at the decision moment because the text length is established.

3.content_age_days: Knowable at the decision moment because the publication date is fixed.

4.days_since_last_update: Knowable at the decision moment from revision logs.

5.gsc_position_tier: Knowable at the decision moment based on historical average ranking brackets.

In [3]:
# Query 2: Build the slice with our exact features, label, and exclusion rule
query_features = f"""
SELECT
    d.content_type,
    d.word_count,
    f.gsc_avg_position,
    (f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0)) AS ctr,
    f.gsc_impressions,
    CASE
        WHEN f.gsc_avg_position <= 10.0 AND (f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0)) < 0.05 THEN 1
        ELSE 0
    END as needs_redesign
FROM '{hf_path}' AS f
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' AS d
  ON f.content_hash_id = d.content_hash_id
WHERE f.gsc_impressions > 0
"""
df = con.execute(query_features).df()
print("Data slice loaded successfully. First 5 rows:")
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data slice loaded successfully. First 5 rows:


,content_type,word_count,gsc_avg_position,ctr,gsc_impressions,needs_redesign
0,keyword article,2855,3.500000,0.0,4,1
1,keyword article,3281,5.875000,0.0,8,1
2,keyword article,3579,4.333333,0.0,12,1
3,keyword article,2993,1.400000,0.0,5,1
4,keyword article,3500,3.476190,0.0,21,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Results:

1.Grain: The query confirms the unique key is report_date + content_hash_id. One row strictly equals one page's performance for one specific day.

2.Date Span: The data is perfectly bounded from 2026-03-01 to 2026-03-31 (exactly one month).

3.Availability: Out of ~9.8 million total rows, only ~3.6 million rows actually have GSC data available.

In [4]:
# 1. Verify the grain: Checking if one row = one page per day
print("1. Verifying Grain (Is report_date + content_hash_id unique?):")
query_grain_check = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT report_date || content_hash_id) as unique_keys
FROM '{hf_path}'
"""
display(con.execute(query_grain_check).df())

# 2. Verify the date span and row count for our slice
print("\n2. Verifying Date Span & Row Count:")
query_span = f"""
SELECT
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(*) as slice_rows
FROM '{hf_path}'
"""
display(con.execute(query_span).df())

# 3. Verify Availability: How many rows survive when GSC data is actually available?
print("\n3. Verifying Availability (gsc_data_available IS TRUE):")
query_availability = f"""
SELECT COUNT(*) as usable_rows
FROM '{hf_path}'
WHERE gsc_data_available IS TRUE
"""
display(con.execute(query_availability).df())

1. Verifying Grain (Is report_date + content_hash_id unique?):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_keys
0,9841378,9841378



2. Verifying Date Span & Row Count:


,min_date,max_date,slice_rows
0,2026-03-01,2026-03-31,9841378



3. Verifying Availability (gsc_data_available IS TRUE):


,usable_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitation: This data can never tell us the complete on-page engagement story for every URL. Because clients adopt tracking tools at different times, there is an unbalanced history: many rows have Google Search Console (GSC) data showing how they rank, but completely lack Google Analytics (GA4) data to show how users actually behave once they click. We are blind to the UX metrics for those specific pages.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 4: Checking the limitation of missing data overlays (GSC vs GA4)
print("Data Limits: Checking the overlap of GSC and GA4 data")
query_limits = f"""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) as has_gsc,
    SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) as has_ga4,
    SUM(CASE WHEN gsc_data_available AND ga4_data_available THEN 1 ELSE 0 END) as has_both
FROM '{hf_path}'
"""
display(con.execute(query_limits).df())

Data Limits: Checking the overlap of GSC and GA4 data


,total_rows,has_gsc,has_ga4,has_both
0,9841378,3611061.0,413966.0,364347.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.